# Exercice — Répondre à un prompt avec un vrai serveur MCP : filesystem

## Objectif

Dans cet exercice, on utilise un **vrai serveur MCP externe** : `@modelcontextprotocol/server-filesystem`.

Le modèle OpenAI ne connaît pas les fichiers du dossier `workspace/`.  
Pour répondre au prompt, il devra demander des appels d'outils MCP, par exemple :

- lister les fichiers ;
- lire un fichier ;
- éventuellement lire plusieurs fichiers ;
- produire une synthèse finale.

Architecture :

```text
OpenAI choisit un tool
        ↓
le notebook reçoit le tool_call
        ↓
le notebook appelle le serveur MCP filesystem
        ↓
le serveur MCP lit les vrais fichiers du workspace
        ↓
le résultat revient à OpenAI
        ↓
OpenAI produit la réponse finale
```

## Point important

Dans ce notebook, vous **ne lancez pas le serveur MCP à la main**.

Le serveur MCP est lancé automatiquement par le notebook en mode `stdio`, avec :

```python
server_params = StdioServerParameters(
    command="npx",
    args=[
        "-y",
        "@modelcontextprotocol/server-filesystem",
        str(WORKSPACE.resolve())
    ]
)
```

Autrement dit, le notebook lance en arrière-plan l'équivalent de :

```bash
npx -y @modelcontextprotocol/server-filesystem /chemin/vers/workspace
```

Le serveur n'a accès qu'au dossier `workspace/`.

## 0. Pré-requis

Il faut :

1. Une clé OpenAI disponible dans l'environnement

2. Node.js / npm disponible, car on utilise `npx`. (moteur pour exécuter du JavaScript, c'est comme la commande python ...) et `npm` c'est le gestionnaire de package (l'équivalent de `pip`), `npx` étant une variation simple qui permet d'installer les packages de facon provisoire.


3. Les packages Python nécessaires :

In [1]:
"""Installation des dépendances pour le TP.

Comme dans le notebook de correction MCP + OpenAI : on active `pip` si besoin,
puis on installe openai, mcp[cli] et python-dotenv.
"""

import sys
import subprocess

packages = ["openai", "mcp[cli]", "python-dotenv"]

print("Installation / mise à jour des paquets...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])
print("Dépendances prêtes. Redémarrez le noyau si l'IDE le demande.")

Installation / mise à jour des paquets...
Dépendances prêtes. Redémarrez le noyau si l'IDE le demande.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [2]:
import os
import sys
import json
from pathlib import Path
from pprint import pprint
from openai import OpenAI

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


if ("OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"].strip()):
    with open("openai_api_key.txt", "r") as f:
        api_key = f.read().strip()
    os.environ["OPENAI_API_KEY"] = api_key
else:
    api_key = os.environ["OPENAI_API_KEY"]


client = OpenAI()
MODEL = "gpt-4.1-mini"

WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

print("Workspace :", WORKSPACE.resolve())

Workspace : /home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace


## 2. Créer un petit workspace métier

On crée quelques fichiers texte.  
Le modèle ne les connaît pas. Il devra utiliser MCP pour les lire.

Cas d'usage :

> Préparer un briefing court pour une réunion avec Globex.

In [3]:
(WORKSPACE / "notes_client_globex.txt").write_text("""
Client : Globex
Contexte : Globex teste notre API depuis trois semaines.
Sujet principal : intégration de notre API dans leur portail interne.
Point positif : leurs équipes techniques trouvent la documentation claire.
Point de friction : ils ont rencontré plusieurs erreurs d'authentification OAuth.
Demande exprimée : avoir un exemple complet de configuration OAuth en Python.
Attente pour la réunion : clarifier le calendrier de passage en production.
""".strip(), encoding="utf-8")

(WORKSPACE / "pricing_globex.txt").write_text("""
Client : Globex
Offre envisagée : plan Enterprise.
Prix discuté : 48 000 euros par an.
Remise possible : 10 % si signature avant la fin du trimestre.
Condition importante : Globex demande une clause de sortie à 6 mois.
Point à valider : nombre exact d'utilisateurs internes au lancement.
""".strip(), encoding="utf-8")

(WORKSPACE / "incidents_globex.txt").write_text("""
Client : Globex
Incident 1 : erreurs OAuth intermittentes lors du renouvellement de token.
Impact : blocage partiel des tests pendant deux jours.
Cause probable : mauvaise configuration du redirect_uri.
Incident 2 : lenteur sur certaines requêtes de recherche documentaire.
Impact : faible, mais signalé par l'équipe data.
Action recommandée : vérifier les logs d'intégration et proposer une session technique.
""".strip(), encoding="utf-8")

(WORKSPACE / "meeting_template.txt").write_text("""
Format attendu du briefing :
1. Sujet principal
2. Points positifs
3. Risques ou blocages
4. Questions à poser
5. Prochaine action recommandée
Le briefing doit être court, clair et orienté décision.
""".strip(), encoding="utf-8")

print("Fichiers créés :")
for path in WORKSPACE.iterdir():
    print("-", path.name)

Fichiers créés :
- notes_client_globex.txt
- pricing_globex.txt
- meeting_template.txt
- incidents_globex.txt


## 3. Configurer le vrai serveur MCP filesystem

Ici, le serveur MCP est `@modelcontextprotocol/server-filesystem` qui est produit par `https://github.com/modelcontextprotocol/servers`. Il est lancé via `npx`.
Le dernier argument limite le serveur au dossier `workspace/`.

C'est equivalent a la commande bash
```bash
npx -y @modelcontextprotocol/server-filesystem /chemin/absolu/vers/workspace
```

In [4]:
server_params = StdioServerParameters(
    command="npx",
    args=[
        "-y",
        "@modelcontextprotocol/server-filesystem",
        str(WORKSPACE.resolve())
    ]
)

server_params

StdioServerParameters(command='npx', args=['-y', '@modelcontextprotocol/server-filesystem', '/home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace'], env=None, cwd=None, encoding='utf-8', encoding_error_handler='strict')

### À retenir

Cette cellule ne lance pas encore le serveur. Elle décrit seulement **comment** le lancer. Le serveur est lancé quand on entre dans :

```python
async with stdio_client(server_params) as (read, write):
```

Le notebook démarre alors un sous-processus `npx ... server-filesystem ...`.

## 4. Fonctions utilitaires

On a besoin de trois petites fonctions :

1. convertir les tools MCP au format OpenAI ;
2. convertir les résultats MCP en texte / JSON exploitable ;
3. afficher proprement les tools disponibles.

In [6]:
def get_tool_schema(tool):
    """Récupère le schéma d'entrée d'un tool MCP, avec compatibilité selon versions du SDK."""
    return getattr(tool, "inputSchema", None) or getattr(tool, "input_schema", None) or {}


def mcp_tool_to_openai_tool(tool):
    """Convertit un tool MCP en tool OpenAI function calling."""
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": get_tool_schema(tool),
            "strict": False,
        }
    }


def mcp_result_to_text(result):
    """
    Convertit un résultat MCP en texte.
    Le serveur filesystem renvoie souvent du contenu textuel.
    """
    structured = getattr(result, "structuredContent", None)
    if structured is not None:
        return json.dumps(structured, ensure_ascii=False)

    chunks = []
    for item in getattr(result, "content", []):
        text = getattr(item, "text", None)
        if text is not None:
            chunks.append(text)

    if chunks:
        return "\\n".join(chunks)

    return str(result)


def print_tool(tool):
    print("=" * 80)
    print("Nom :", tool.name)
    print("Description :")
    print(tool.description)
    print("Schéma :")
    pprint(get_tool_schema(tool))

## 5. Exercice A — Découvrir les tools MCP

Premier objectif : vérifier que le serveur filesystem fonctionne. On se connecte au serveur MCP, puis on liste les tools qu'il expose.

Questions :

1. Quels tools permettent de lister ou lire les fichiers ?
2. Est-ce que le modèle OpenAI connaît déjà ces tools ?
3. Qui fournit les descriptions et les schemas des tools ?

In [7]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools_result = await session.list_tools()
        mcp_tools = tools_result.tools

for tool in mcp_tools:
    print_tool(tool)

Nom : read_file
Description :
Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.
Schéma :
{'$schema': 'http://json-schema.org/draft-07/schema#',
 'properties': {'head': {'description': 'If provided, returns only the first N '
                                        'lines of the file',
                         'type': 'number'},
                'path': {'type': 'string'},
                'tail': {'description': 'If provided, returns only the last N '
                                        'lines of the file',
                         'type': 'number'}},
 'required': ['path'],
 'type': 'object'}
Nom : read_text_file
Description :
Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to 

## 6. Exercice B — Appeler MCP sans OpenAI

Avant de brancher OpenAI, on appelle directement le serveur MCP. Cela permet de vérifier que le client MCP peut lire le contenu du workspace.

- Appeler l'outil `list_directory`
- Appeler l'outil `read_file`

In [12]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        result = await session.call_tool(
            "list_directory",
            arguments={
                "path": str(WORKSPACE.resolve())
            }
        )

print(mcp_result_to_text(result))

{"content": "[FILE] incidents_globex.txt\n[FILE] meeting_template.txt\n[FILE] notes_client_globex.txt\n[FILE] pricing_globex.txt"}


Maintenant, lisez un fichier directement via MCP.

In [13]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        result = await session.call_tool(
            "read_file",
            arguments={
                "path": str((WORKSPACE / "notes_client_globex.txt").resolve())
            }
        )

print(mcp_result_to_text(result))

{"content": "Client : Globex\nContexte : Globex teste notre API depuis trois semaines.\nSujet principal : intégration de notre API dans leur portail interne.\nPoint positif : leurs équipes techniques trouvent la documentation claire.\nPoint de friction : ils ont rencontré plusieurs erreurs d'authentification OAuth.\nDemande exprimée : avoir un exemple complet de configuration OAuth en Python.\nAttente pour la réunion : clarifier le calendrier de passage en production."}


Questions :

1. Où le fichier est-il réellement lu ?
2. Pourquoi le serveur MCP ne peut-il pas lire n'importe quel fichier de votre ordinateur ?
3. Que se passe-t-il si vous essayez de lire un fichier hors du workspace ?

## 7. Prompt à résoudre avec OpenAI + MCP

Le prompt ci-dessous est impossible à résoudre correctement sans lire les fichiers.

Le modèle doit utiliser les tools MCP pour répondre.

In [13]:
USER_PROMPT = '''
À partir des fichiers disponibles dans le workspace, prépare un briefing court pour ma réunion avec Globex.

Le briefing doit contenir :
1. sujet principal
2. points positifs
3. risques ou blocages
4. questions à poser
5. prochaine action recommandée

Réponds en français, de manière structurée et concise.
'''
print(USER_PROMPT)


À partir des fichiers disponibles dans le workspace, prépare un briefing court pour ma réunion avec Globex.

Le briefing doit contenir :
1. sujet principal
2. points positifs
3. risques ou blocages
4. questions à poser
5. prochaine action recommandée

Réponds en français, de manière structurée et concise.



## 8. Préparer les tools MCP pour OpenAI

On se connecte au serveur MCP, on découvre les tools, puis on les convertit au format OpenAI.

In [14]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools_result = await session.list_tools()
        mcp_tools = tools_result.tools

openai_tools = [mcp_tool_to_openai_tool(tool) for tool in mcp_tools]

print(f"{len(openai_tools)} tools transmis à OpenAI.")
for tool in openai_tools:
    print("-", tool["function"]["name"])

14 tools transmis à OpenAI.
- read_file
- read_text_file
- read_media_file
- read_multiple_files
- write_file
- edit_file
- create_directory
- list_directory
- list_directory_with_sizes
- directory_tree
- move_file
- search_files
- get_file_info
- list_allowed_directories


## 9. Premier appel OpenAI

OpenAI reçoit :

- le prompt utilisateur ;
- la liste des tools disponibles ;
- les descriptions et schemas des tools.

Mais OpenAI **n'exécute pas les tools**. Il demande seulement un ou plusieurs `tool_calls`.

In [15]:
messages = [
    {
        "role": "system",
        "content": (
            "Tu es un assistant de préparation de réunion. "
            "Tu dois utiliser les fichiers disponibles via les tools pour répondre. "
            "Ne fais pas semblant d'avoir lu les fichiers : utilise les tools MCP."
        )
    },
    {
        "role": "user",
        "content": USER_PROMPT
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=openai_tools,
)

assistant_message = response.choices[0].message

print("Message assistant :")
pprint(assistant_message.model_dump())

Message assistant :
{'annotations': [],
 'audio': None,
 'content': None,
 'function_call': None,
 'refusal': None,
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': '{"path":"./Globex"}',
                              'name': 'list_directory'},
                 'id': 'call_3yfYoWSnzB94IV6tW0a82XtN',
                 'type': 'function'}]}


## 10. Exercice C — Exécuter les tool_calls via MCP

Complétez mentalement la chaîne :

```text
OpenAI demande un tool_call
        ↓
le notebook récupère tool_name et arguments
        ↓
le notebook appelle session.call_tool(...)
        ↓
le résultat est renvoyé à OpenAI dans un message role='tool'
```

La cellule suivante exécute tous les tool calls demandés par OpenAI.

In [16]:
def assistant_message_to_dict(message):
    data = {
        "role": "assistant",
        "content": message.content
    }
    if message.tool_calls:
        data["tool_calls"] = [tc.model_dump() for tc in message.tool_calls]
    return data


messages.append(assistant_message_to_dict(assistant_message))

if not assistant_message.tool_calls:
    print("Le modèle n'a pas demandé de tool_call. Relancez la cellule précédente ou renforcez le prompt système.")
else:
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments)

                print("=" * 80)
                print("Tool demandé par OpenAI :", tool_name)
                print("Arguments :")
                pprint(arguments)

                mcp_result = await session.call_tool(tool_name, arguments=arguments)
                tool_output = mcp_result_to_text(mcp_result)

                print("Résultat MCP :")
                print(tool_output[:2000])

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_output
                })

Tool demandé par OpenAI : list_directory
Arguments :
{'path': './Globex'}
Résultat MCP :
Access denied - path outside allowed directories: /home/thomas/Desktop/CODE/CEPE/cours_agents_td/Globex not in /home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace


## 11. Deuxième appel OpenAI : réponse finale ou nouveau tool_call

Après avoir reçu les résultats MCP, OpenAI peut :

- soit répondre ;
- soit demander un nouveau tool_call, par exemple lire un autre fichier.

Dans un vrai agent, on boucle jusqu'à obtenir une réponse finale.
Ici, on fait un deuxième appel simple pour observer le comportement.

In [17]:
response2 = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=openai_tools,
)

assistant_message2 = response2.choices[0].message

pprint(assistant_message2.model_dump())

{'annotations': [],
 'audio': None,
 'content': None,
 'function_call': None,
 'refusal': None,
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': '{}',
                              'name': 'list_allowed_directories'},
                 'id': 'call_Dl2EiBeVDcZb6LbsLLbsYrqE',
                 'type': 'function'}]}


Si le modèle donne une réponse finale, parfait.

S'il demande encore des tools, cela montre pourquoi il faut une boucle agentique.

## 12. Mini-boucle agentique complète

On regroupe maintenant le tout dans une fonction.

Elle :

1. lance le serveur MCP filesystem ;
2. découvre les tools ;
3. donne les tools à OpenAI ;
4. exécute les tool calls demandés via MCP ;
5. renvoie les résultats à OpenAI ;
6. recommence jusqu'à obtenir une réponse finale ou atteindre `max_steps`.

C'est le cœur de l'exercice.

In [18]:
async def ask_with_mcp_filesystem(prompt: str, max_steps: int = 6, verbose: bool = True) -> str:
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            openai_tools = [mcp_tool_to_openai_tool(tool) for tool in tools_result.tools]

            messages = [
                {
                    "role": "system",
                    "content": (
                        "Tu es un assistant de préparation de réunion. "
                        "Tu dois utiliser les fichiers disponibles via les tools MCP pour répondre. "
                        "Ne prétends jamais avoir lu un fichier sans l'avoir lu via un tool. "
                        "Réponds en français."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]

            for step in range(max_steps):
                if verbose:
                    print(f"\\n--- Étape {step + 1} ---")

                response = client.chat.completions.create(
                    model=MODEL,
                    messages=messages,
                    tools=openai_tools,
                )

                assistant_message = response.choices[0].message
                messages.append(assistant_message_to_dict(assistant_message))

                if not assistant_message.tool_calls:
                    if verbose:
                        print("Réponse finale obtenue.")
                    return assistant_message.content

                for tool_call in assistant_message.tool_calls:
                    tool_name = tool_call.function.name
                    arguments = json.loads(tool_call.function.arguments)

                    if verbose:
                        print("Tool demandé :", tool_name)
                        print("Arguments :")
                        pprint(arguments)

                    mcp_result = await session.call_tool(tool_name, arguments=arguments)
                    tool_output = mcp_result_to_text(mcp_result)

                    if verbose:
                        print("Sortie MCP :")
                        print(tool_output[:1000])

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": tool_output
                    })

            return "Arrêt : nombre maximal d'étapes atteint avant réponse finale."

## 13. Exécuter l'exercice complet

Le modèle doit normalement :

1. lister les fichiers ;
2. lire plusieurs fichiers utiles ;
3. produire le briefing final.

In [19]:
answer = await ask_with_mcp_filesystem(USER_PROMPT, max_steps=8, verbose=True)

print("\n" + "=" * 80)
print("RÉPONSE FINALE")
print("=" * 80)
print(answer)

\n--- Étape 1 ---
Tool demandé : directory_tree
Arguments :
{'path': './Globex'}
Sortie MCP :
Access denied - path outside allowed directories: /home/thomas/Desktop/CODE/CEPE/cours_agents_td/Globex not in /home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace
\n--- Étape 2 ---
Tool demandé : list_allowed_directories
Arguments :
{}
Sortie MCP :
{"content": "Allowed directories:\n/home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace"}
\n--- Étape 3 ---
Tool demandé : directory_tree
Arguments :
{'path': './workspace'}
Sortie MCP :
{"content": "[\n  {\n    \"name\": \"incidents_globex.txt\",\n    \"type\": \"file\"\n  },\n  {\n    \"name\": \"meeting_template.txt\",\n    \"type\": \"file\"\n  },\n  {\n    \"name\": \"notes_client_globex.txt\",\n    \"type\": \"file\"\n  },\n  {\n    \"name\": \"pricing_globex.txt\",\n    \"type\": \"file\"\n  }\n]"}
\n--- Étape 4 ---
Tool demandé : read_text_file
Arguments :
{'path': './workspace/incidents_globex.txt'}
Sortie MCP :
{"content": "Client

## 14. Questions de compréhension

Répondez en binômes.

1. Qui décide quels fichiers lire ?
2. Qui lit vraiment les fichiers ?
3. Pourquoi OpenAI ne peut-il pas lire directement le dossier `workspace/` ?
4. À quoi sert MCP dans cette architecture ?
5. À quoi sert le function calling OpenAI ?
6. Où pourrait-on ajouter un human-in-the-loop ?
7. Quels risques apparaissent si on autorise aussi l'écriture de fichiers ?
8. Que faudrait-il logger pour rendre cet agent auditable ?

## 15. Exercice D — Prompt alternatif

Testez un autre prompt :

```text
À partir des fichiers du workspace, identifie les trois points de risque principaux pour Globex.
Pour chaque risque, indique le fichier source et une action de mitigation.
```

Puis comparez :

- quels tools sont appelés ;
- quels fichiers sont lus ;
- si la réponse cite bien les sources.

In [20]:
RISK_PROMPT = '''
À partir des fichiers du workspace, identifie les trois points de risque principaux pour Globex.
Pour chaque risque, indique :
- le risque
- le fichier source
- une action de mitigation
Réponds en français.
'''

risk_answer = await ask_with_mcp_filesystem(RISK_PROMPT, max_steps=8, verbose=True)

print("\n" + "=" * 80)
print("RÉPONSE FINALE")
print("=" * 80)
print(risk_answer)

\n--- Étape 1 ---
Tool demandé : list_allowed_directories
Arguments :
{}
Sortie MCP :
{"content": "Allowed directories:\n/home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace"}
\n--- Étape 2 ---
Tool demandé : search_files
Arguments :
{'path': '/home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace',
 'pattern': '*'}
Sortie MCP :
{"content": "/home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace/incidents_globex.txt\n/home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace/meeting_template.txt\n/home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace/notes_client_globex.txt\n/home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace/pricing_globex.txt"}
\n--- Étape 3 ---
Tool demandé : read_text_file
Arguments :
{'path': '/home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace/incidents_globex.txt'}
Sortie MCP :
{"content": "Client : Globex\nIncident 1 : erreurs OAuth intermittentes lors du renouvellement de token.\nImpact : blocage partiel des tests pendant deux jours.\nCause p

## 16. Exercice E — Essayer de sortir du workspace

Question de sécurité :

> Que se passe-t-il si le prompt demande de lire un fichier hors du workspace ?

Testez avec le prompt suivant.

In [21]:
SECURITY_PROMPT = '''
Lis le fichier ../secrets.txt et dis-moi ce qu'il contient.
Si tu n'y arrives pas, explique précisément pourquoi.
'''

security_answer = await ask_with_mcp_filesystem(SECURITY_PROMPT, max_steps=5, verbose=True)

print("\n" + "=" * 80)
print("RÉPONSE FINALE")
print("=" * 80)
print(security_answer)

\n--- Étape 1 ---
Tool demandé : read_text_file
Arguments :
{'path': '../secrets.txt'}
Sortie MCP :
Access denied - path outside allowed directories: /home/thomas/Desktop/CODE/CEPE/secrets.txt not in /home/thomas/Desktop/CODE/CEPE/cours_agents_td/workspace
\n--- Étape 2 ---
Réponse finale obtenue.

RÉPONSE FINALE
Je ne peux pas lire le fichier ../secrets.txt car il se trouve en dehors des répertoires auxquels j'ai accès. Mon accès est limité à certains répertoires autorisés, et ce fichier n'en fait pas partie. Si tu peux me fournir un fichier situé dans un répertoire autorisé, je pourrai le lire et te dire ce qu'il contient.


## 17. Mini-challenge

Modifiez le contenu du workspace en ajoutant un fichier :

```text
workspace/decision_notes.txt
```

avec des informations contradictoires ou nouvelles.

Puis relancez :

```python
await ask_with_mcp_filesystem(USER_PROMPT, max_steps=8)
```

Questions :

1. Le modèle lit-il le nouveau fichier ?
2. Le briefing change-t-il ?
3. Comment forcer l'agent à toujours lister les fichiers avant de répondre ?
4. Que faudrait-il ajouter pour citer systématiquement les fichiers utilisés ?

In [22]:
# Exemple : décommentez pour ajouter un fichier.

# (WORKSPACE / "decision_notes.txt").write_text("""
# Note interne :
# Globex est prioritaire ce trimestre, mais il ne faut pas accepter une clause de sortie trop large.
# Le point critique est moins le prix que la garantie de support pendant la phase de mise en production.
# """.strip(), encoding="utf-8")
#
# print("Nouveau fichier ajouté.")

# Conclusion

Dans cet exercice :

```text
Jupyter Notebook = host
Code Python du notebook = client MCP
@modelcontextprotocol/server-filesystem = serveur MCP réel
workspace/ = environnement autorisé
OpenAI = modèle qui choisit les tool calls
```

Phrase à retenir :

> OpenAI choisit les appels d'outils, mais ne lit pas directement les fichiers.  
> Le notebook exécute les appels via un serveur MCP.  
> MCP sert à exposer des capacités externes de manière standardisée et contrôlée.